# Flight Delay Prediction — Data Pipeline

This notebook covers the full data pipeline:
1. BTS flight data download
2. NOAA weather download and join
3. Holiday and weekend flags
4. Temporal train/val/test split

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import time
import os
import gc
import shutil
import glob
import requests
from io import StringIO, BytesIO
from zipfile import ZipFile
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import timedelta
import holidays
import pyarrow.parquet as pq
import pyarrow as pa

# For Jupyter notebooks — render plots inline
%matplotlib inline

print('✅ All imports successful!')

✅ All imports successful!


In [2]:
# Set data directory relative to project root
cwd = os.getcwd()
if os.path.basename(cwd) == 'notebooks':
    DRIVE_PATH = os.path.abspath(os.path.join(cwd, '..', 'data'))
else:
    DRIVE_PATH = os.path.abspath(os.path.join(cwd, 'data'))

os.makedirs(DRIVE_PATH, exist_ok=True)
print(f'Data directory: {DRIVE_PATH}')

Data directory: /Users/sripriya/ucsd/MDS/CapstoneProject/AirlineArrivalDelay/data


## Section 1: BTS Flight Data Download

Downloads BTS On-Time Performance data (2018–2024) and saves as parquet.

In [3]:
def load_bts_month(year, month):
    url = (
        f'https://transtats.bts.gov/PREZIP/'
        f'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip'
    )
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    z = ZipFile(BytesIO(response.content))
    csv_name = [f for f in z.namelist() if f.endswith('.csv')][0]
    cols = [
        'Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate',
        'Reporting_Airline', 'Flight_Number_Reporting_Airline',
        'Origin', 'Dest', 'CRSDepTime', 'DepTimeBlk',
        'CRSArrTime', 'ArrDel15', 'CRSElapsedTime', 'Distance', 'DistanceGroup'
    ]
    df = pd.read_csv(z.open(csv_name), low_memory=False, usecols=cols)
    df['Year'] = year
    df['Month'] = month
    return df

tasks = [(y, m) for y in range(2018, 2025) for m in range(1, 13)]
frames, failed = [], []

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(load_bts_month, y, m): (y, m) for y, m in tasks}
    for future in as_completed(futures):
        y, m = futures[future]
        try:
            chunk = future.result()
            frames.append(chunk)
            print(f'✅ {y}-{m:02d}: {len(chunk):,} rows')
        except Exception as e:
            print(f'⚠️  {y}-{m:02d}: Skipped ({e})')
            failed.append((y, m))

bts = pd.concat(frames, ignore_index=True).sort_values('FlightDate').reset_index(drop=True)
print(f'\n🎉 Total: {len(bts):,} rows × {bts.shape[1]} columns')
print(f'📅 Range: {bts["FlightDate"].min()} → {bts["FlightDate"].max()}')

bts.to_parquet(f'{DRIVE_PATH}/flights_2018_2024_v2.parquet', index=False)
print(f'💾 Saved to {DRIVE_PATH}/flights_2018_2024_v2.parquet')

✅ 2018-03: 612,034 rows
✅ 2018-01: 570,138 rows
✅ 2018-04: 596,078 rows
✅ 2018-02: 520,769 rows
✅ 2018-06: 626,217 rows
✅ 2018-07: 645,317 rows
✅ 2018-05: 616,573 rows
✅ 2018-08: 637,103 rows
✅ 2018-09: 585,765 rows
✅ 2018-10: 616,128 rows
✅ 2018-12: 593,842 rows
✅ 2018-11: 586,231 rows
✅ 2019-02: 533,175 rows
✅ 2019-01: 583,985 rows
✅ 2019-04: 612,023 rows
✅ 2019-03: 632,074 rows
✅ 2019-07: 659,029 rows
✅ 2019-05: 636,390 rows
✅ 2019-06: 636,691 rows
✅ 2019-08: 658,461 rows
✅ 2019-10: 636,014 rows
✅ 2019-09: 605,979 rows
✅ 2019-11: 602,453 rows
✅ 2019-12: 625,763 rows
✅ 2020-01: 607,346 rows
✅ 2020-02: 574,268 rows
✅ 2020-05: 180,617 rows
✅ 2020-04: 313,382 rows
✅ 2020-06: 223,732 rows
✅ 2020-03: 648,229 rows
✅ 2020-07: 352,888 rows
✅ 2020-08: 376,715 rows
✅ 2020-09: 323,347 rows
✅ 2020-12: 371,357 rows
✅ 2020-11: 364,367 rows
✅ 2020-10: 352,106 rows
✅ 2021-01: 361,428 rows
✅ 2021-02: 332,468 rows
✅ 2021-04: 450,637 rows
✅ 2021-03: 444,476 rows
✅ 2021-05: 495,544 rows
✅ 2021-06: 546,1

## Section 2: Weather Data Download & Join

Downloads NOAA ASOS hourly weather per airport/year and joins with BTS at T-2 hours before scheduled departure.

In [4]:
bts = pd.read_parquet(f'{DRIVE_PATH}/flights_2018_2024_v2.parquet')
print(f'BTS loaded: {bts.shape}')
bts.head()

BTS loaded: (45968068, 17)


,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Flight_Number_Reporting_Airline,Origin,Dest,CRSDepTime,DepTimeBlk,CRSArrTime,ArrDel15,CRSElapsedTime,Distance,DistanceGroup
0,2018,1,1,1,1,2018-01-01,AS,54.0,PDX,EWR,633,0600-0659,1439,0.0,306.0,2434.0,10
1,2018,1,1,1,1,2018-01-01,DL,16.0,DEN,DTW,955,0900-0959,1453,0.0,178.0,1123.0,5
2,2018,1,1,1,1,2018-01-01,DL,26.0,ATL,MCO,1955,1900-1959,2131,1.0,96.0,404.0,2
3,2018,1,1,1,1,2018-01-01,DL,27.0,MCO,ATL,830,0800-0859,1004,0.0,94.0,404.0,2
4,2018,1,1,1,1,2018-01-01,DL,29.0,ATL,MCO,1655,1600-1659,1827,0.0,92.0,404.0,2


In [5]:
# Get all unique airport codes (both origin and destination)
airports = sorted(set(bts['Origin'].unique()) | set(bts['Dest'].unique()))
print(f'Total unique airports: {len(airports)}')
print(airports[:10])  # Preview

Total unique airports: 384
['ABE', 'ABI', 'ABQ', 'ABR', 'ABY', 'ACK', 'ACT', 'ACV', 'ACY', 'ADK']


In [6]:
# ============================================================
# CONFIGURATION
# ============================================================
YEARS        = list(range(2018, 2025))     # 2018, 2019, ... 2024
TIMEZONE     = 'America/New_York'
SAVE_DIR     = os.path.join(DRIVE_PATH, 'weather_cache')
DELAY        = 5
OUTPUT_FILE  = os.path.join(DRIVE_PATH, 'bts_with_weather.parquet')

total_jobs = len(airports) * len(YEARS)
print(f"📋 BTS records:     {len(bts):,}")
print(f"✈️  Unique airports: {len(airports)}")
print(f"📅 Years:           {YEARS[0]}–{YEARS[-1]} ({len(YEARS)} years)")
print(f"📦 Total downloads: {len(airports)} airports × {len(YEARS)} years = {total_jobs}")
print(f"⏱️  Estimated time:  ~{(total_jobs * DELAY) // 60} minutes (first run)")
print()

📋 BTS records:     45,968,068
✈️  Unique airports: 384
📅 Years:           2018–2024 (7 years)
📦 Total downloads: 384 airports × 7 years = 2688
⏱️  Estimated time:  ~224 minutes (first run)



In [7]:
# ============================================================
# STEP 1: Download function with retry logic
# ============================================================
def fetch_weather(station, year, max_retries=3):
    url = (
        "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?"
        f"station={station}&"
        "data=tmpf&data=dwpf&data=relh&data=sknt&data=gust&"
        "data=vsby&data=p01i&data=wxcodes&data=feel&"
        f"tz={TIMEZONE}&"
        "format=onlycomma&latlon=no&missing=empty&"
        "trace=0.0001&"
        f"year1={year}&month1=1&day1=1&"
        f"year2={year}&month2=12&day2=31"
    )

    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=60)

            if response.status_code == 429:
                wait_time = (attempt + 1) * 30
                print(f"    ⏳ 429 Rate limited. Waiting {wait_time}s "
                      f"(attempt {attempt + 1}/{max_retries})...")
                time.sleep(wait_time)
                continue

            response.raise_for_status()

            df = pd.read_csv(StringIO(response.text))
            if len(df) > 0:
                return df
            return None

        except requests.exceptions.RequestException as e:
            wait_time = (attempt + 1) * 15
            print(f"    ⚠️ Error: {e}. Retrying in {wait_time}s...")
            time.sleep(wait_time)

    return None

In [8]:
files = os.listdir(SAVE_DIR) if os.path.exists(SAVE_DIR) else []
print(f'Cache files: {len(files)}')

In [9]:
# ============================================================
# STEP 2: Download weather — one file per airport per year
# ============================================================
print('=' * 60)
print('DOWNLOADING WEATHER DATA')
print('=' * 60)
os.makedirs(SAVE_DIR, exist_ok=True)
failed_jobs = []
job_num = 0
for year in YEARS:
    print(f'\n--- {year} ---')
    for i, airport in enumerate(airports, 1):
        job_num += 1
        cache_file = os.path.join(SAVE_DIR, f'{airport}_{year}.csv')
        if os.path.exists(cache_file):
            print(f'  [{job_num}/{total_jobs}] 📁 {airport} {year}: cached')
            continue
        print(f'  [{job_num}/{total_jobs}] ⬇️  {airport} {year}...', end=' ')
        df = fetch_weather(airport, year, max_retries=3)
        if df is not None:
            df.to_csv(cache_file, index=False)
            print(f'✅ {len(df):,} records')
        else:
            failed_jobs.append((airport, year))
            print('❌ No data')
        time.sleep(DELAY)
print(f'\n  ✅ Downloaded: {total_jobs - len(failed_jobs)}/{total_jobs}')
print()


DOWNLOADING WEATHER DATA

--- 2018 ---
  [1/2688] ⬇️  ABE 2018... ✅ 114,637 records
  [2/2688] ⬇️  ABI 2018... ✅ 113,695 records
  [3/2688] ⬇️  ABQ 2018... ✅ 112,889 records
  [4/2688] ⬇️  ABR 2018... ✅ 112,091 records
  [5/2688] ⬇️  ABY 2018... ✅ 112,809 records
  [6/2688] ⬇️  ACK 2018... ✅ 113,289 records
  [7/2688] ⬇️  ACT 2018... ✅ 114,773 records
  [8/2688] ⬇️  ACV 2018... ✅ 116,620 records
  [9/2688] ⬇️  ACY 2018... ✅ 115,291 records
  [10/2688] ⬇️  ADK 2018... ❌ No data
  [11/2688] ⬇️  ADQ 2018... ❌ No data
  [12/2688] ⬇️  AEX 2018... ✅ 113,762 records
  [13/2688] ⬇️  AGS 2018... ✅ 114,756 records
  [14/2688] ⬇️  AKN 2018... ❌ No data
  [15/2688] ⬇️  ALB 2018... ✅ 99,582 records
  [16/2688] ⬇️  ALO 2018... ✅ 114,552 records
  [17/2688] ⬇️  ALS 2018... ✅ 112,858 records
  [18/2688] ⬇️  ALW 2018... ✅ 108,883 records
  [19/2688] ⬇️  AMA 2018... ✅ 113,678 records
  [20/2688] ⬇️  ANC 2018... ❌ No data
  [21/2688] ⬇️  APN 2018... ✅ 112,351 records
  [22/2688] ⬇️  ART 2018... ✅ 100,234

In [10]:
# ============================================================
# STEP 3: Retry failed stations with ICAO prefix
# ============================================================
if failed_jobs:
    print('=' * 60)
    print(f'RETRYING {len(failed_jobs)} FAILED DOWNLOADS')
    print('=' * 60)

    still_failed = []

    for airport, year in failed_jobs:
        cache_file = os.path.join(SAVE_DIR, f'{airport}_{year}.csv')
        print(f'  🔄 Retrying {airport} {year} as K{airport}...', end=' ')
        time.sleep(10)

        df = fetch_weather(f'K{airport}', year, max_retries=3)

        if df is not None:
            df['station'] = airport
            df.to_csv(cache_file, index=False)
            print(f'✅ Recovered ({len(df):,} records)')
        else:
            still_failed.append((airport, year))
            print('❌ Permanently failed')

    if still_failed:
        failed_airports = sorted(set(a for a, y in still_failed))
        print(f'\n  ⛔ No data available for: {failed_airports}')
        print(f'     Total failed: {len(still_failed)}')
    print()
else:
    still_failed = []
    print('No failed downloads 🎉\n')

RETRYING 293 FAILED DOWNLOADS
  🔄 Retrying ADK 2018 as KADK... ❌ Permanently failed
  🔄 Retrying ADQ 2018 as KADQ... ❌ Permanently failed
  🔄 Retrying AKN 2018 as KAKN... ❌ Permanently failed
  🔄 Retrying ANC 2018 as KANC... ❌ Permanently failed
  🔄 Retrying AZA 2018 as KAZA... ❌ Permanently failed
  🔄 Retrying BET 2018 as KBET... ❌ Permanently failed
  🔄 Retrying BKG 2018 as KBKG... ❌ Permanently failed
  🔄 Retrying BQN 2018 as KBQN... ❌ Permanently failed
  🔄 Retrying BRW 2018 as KBRW... ❌ Permanently failed
  🔄 Retrying CDB 2018 as KCDB... ❌ Permanently failed
  🔄 Retrying CDV 2018 as KCDV... ❌ Permanently failed
  🔄 Retrying DLG 2018 as KDLG... ❌ Permanently failed
  🔄 Retrying FAI 2018 as KFAI... ❌ Permanently failed
  🔄 Retrying FCA 2018 as KFCA... ❌ Permanently failed
  🔄 Retrying GST 2018 as KGST... ❌ Permanently failed
  🔄 Retrying GUM 2018 as KGUM... ❌ Permanently failed
  🔄 Retrying HHH 2018 as KHHH... ❌ Permanently failed
  🔄 Retrying HNL 2018 as KHNL... ❌ Permanently faile

In [11]:
# ============================================================
# STEP 4: Combine and clean weather data (from disk, memory-safe)
# ============================================================
print('=' * 60)
print('CLEANING WEATHER DATA')
print('=' * 60)
csv_files = sorted([os.path.join(SAVE_DIR, f) for f in os.listdir(SAVE_DIR) if f.endswith('.csv')])
print(f'  Reading {len(csv_files)} cached files...')
numeric_cols = ['tmpf', 'dwpf', 'relh', 'sknt', 'gust', 'vsby', 'p01i', 'feel']
agg_chunks = []
for i, f in enumerate(csv_files):
    df = pd.read_csv(f)
    df['valid'] = pd.to_datetime(df['valid'], errors='coerce')
    df = df.dropna(subset=['valid'])
    df['date'] = df['valid'].dt.date.astype(str)
    df['hour'] = df['valid'].dt.hour
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    agg = df.groupby(['station', 'date', 'hour']).agg(
        temp_f       = ('tmpf', 'mean'),
        dewpoint_f   = ('dwpf', 'mean'),
        humidity     = ('relh', 'mean'),
        feels_like_f = ('feel', 'mean'),
        wind_kts     = ('sknt', 'mean'),
        gust_kts     = ('gust', 'max'),
        visibility   = ('vsby', 'mean'),
        precip_in    = ('p01i', 'sum'),
        wx_codes     = ('wxcodes', lambda x: '|'.join(x.dropna().unique())),
    ).reset_index()
    agg_chunks.append(agg)
    if (i + 1) % 200 == 0:
        print(f'  Processed {i+1}/{len(csv_files)} files...')
weather_hourly = pd.concat(agg_chunks, ignore_index=True)
del agg_chunks
gc.collect()
print(f'  Hourly records: {len(weather_hourly):,}')
print(f'  Stations: {weather_hourly["station"].nunique()}')
print(f'  Date range: {weather_hourly["date"].min()} → {weather_hourly["date"].max()}')
print()

CLEANING WEATHER DATA
  Reading 2395 cached files...
  Processed 200/2395 files...
  Processed 400/2395 files...
  Processed 600/2395 files...
  Processed 800/2395 files...
  Processed 1000/2395 files...
  Processed 1200/2395 files...
  Processed 1400/2395 files...
  Processed 1600/2395 files...
  Processed 1800/2395 files...
  Processed 2000/2395 files...
  Processed 2200/2395 files...
  Hourly records: 20,842,501
  Stations: 343
  Date range: 2018-01-01 → 2024-12-30



In [12]:
# ============================================================
# STEP 5: Engineer weather features
# ============================================================
print("=" * 60)
print("ENGINEERING WEATHER FEATURES")
print("=" * 60)

weather_hourly['is_rain'] = weather_hourly['wx_codes'].str.contains(
    'RA|DZ|TS', case=False, na=False).astype(int)

weather_hourly['is_snow'] = weather_hourly['wx_codes'].str.contains(
    'SN|SG|PL|IC', case=False, na=False).astype(int)

weather_hourly['is_fog'] = weather_hourly['wx_codes'].str.contains(
    'FG|BR|HZ', case=False, na=False).astype(int)

weather_hourly['low_visibility'] = (
    weather_hourly['visibility'] < 3).astype(int)

weather_hourly['high_wind'] = (
    weather_hourly['gust_kts'] > 25).astype(int)

weather_hourly['severe_weather'] = (
    weather_hourly['is_rain']        |
    weather_hourly['is_snow']        |
    weather_hourly['low_visibility'] |
    weather_hourly['high_wind']
).astype(int)

weather_hourly.to_parquet(os.path.join(DRIVE_PATH, 'weather_hourly_all_airports.parquet'), index=False)

print("  ✅ Features: is_rain, is_snow, is_fog, "
      "low_visibility, high_wind, severe_weather")
print(f"  💾 Saved: {DRIVE_PATH}/weather_hourly_all_airports.parquet")
print()


ENGINEERING WEATHER FEATURES
  ✅ Features: is_rain, is_snow, is_fog, low_visibility, high_wind, severe_weather
  💾 Saved: /Users/sripriya/ucsd/MDS/CapstoneProject/AirlineArrivalDelay/data/weather_hourly_all_airports.parquet



In [13]:
# ============================================================
# LOAD SAVED WEATHER DATA
# ============================================================
print("Loading saved weather data...")

weather_hourly = pd.read_parquet(os.path.join(DRIVE_PATH, 'weather_hourly_all_airports.parquet'))

print(f"  ✅ Loaded: {len(weather_hourly):,} hourly records")
print(f"  Stations: {weather_hourly['station'].nunique()}")
print(f"  Date range: {weather_hourly['date'].min()} → {weather_hourly['date'].max()}")
print(f"  Columns: {list(weather_hourly.columns)}")
print()

OUTPUT_FILE = os.path.join(DRIVE_PATH, 'bts_with_weather.parquet')

In [14]:
# ============================================================
# STEP 6: Prepare BTS join keys
# ============================================================
print("=" * 60)
print("STEP 6: PREPARING BTS MERGE KEYS")
print("=" * 60)

bts['FlightDate']  = pd.to_datetime(bts['FlightDate'])
bts['date']        = bts['FlightDate'].dt.date.astype(str)
bts['dep_hour']    = (bts['CRSDepTime'] // 100).astype('Int64')
bts['arr_hour']    = (bts['CRSArrTime'] // 100).astype('Int64')

# ── New columns: scheduled times minus 2 hours (clamped to 0–23) ──
bts['dep_hour_minus2'] = ((bts['dep_hour'] - 2) % 24).astype('Int64')
bts['arr_hour_minus2'] = ((bts['arr_hour'] - 2) % 24).astype('Int64')

print(f"  BTS records:  {len(bts):,}")
print(f"  Date range:   {bts['FlightDate'].min()} → {bts['FlightDate'].max()}")
print(f"  dep_hour sample (orig → -2h): "
      f"{bts['dep_hour'].iloc[0]} → {bts['dep_hour_minus2'].iloc[0]}")
print(f"  arr_hour sample (orig → -2h): "
      f"{bts['arr_hour'].iloc[0]} → {bts['arr_hour_minus2'].iloc[0]}")
print()

STEP 6: PREPARING BTS MERGE KEYS
  BTS records:  45,968,068
  Date range:   2018-01-01 00:00:00 → 2024-12-31 00:00:00
  dep_hour sample (orig → -2h): 6 → 4
  arr_hour sample (orig → -2h): 14 → 12



In [15]:
# ============================================================
# STEP 7: Merge ORIGIN weather
# ============================================================
print("=" * 60)
print("STEP 7: MERGING ORIGIN WEATHER")
print("=" * 60)

origin_wx = weather_hourly.copy()
origin_wx.columns = ['station', 'date', 'hour'] + \
    [f'origin_{c}' for c in origin_wx.columns[3:]]

bts_merged = bts.merge(
    origin_wx,
    left_on  = ['Origin', 'date', 'dep_hour_minus2'],
    right_on = ['station', 'date', 'hour'],
    how      = 'left'
).drop(columns=['station', 'hour'])

print(f"  Match rate: {bts_merged['origin_temp_f'].notna().mean():.1%}")
print()

STEP 7: MERGING ORIGIN WEATHER
  Match rate: 96.5%



In [16]:
# ============================================================
# STEP 8: Merge DESTINATION weather
# ============================================================
print("=" * 60)
print("STEP 8: MERGING DESTINATION WEATHER")
print("=" * 60)

dest_wx = weather_hourly.copy()
dest_wx.columns = ['station', 'date', 'hour'] + \
    [f'dest_{c}' for c in dest_wx.columns[3:]]

bts_merged = bts_merged.merge(
    dest_wx,
    left_on  = ['Dest', 'date', 'arr_hour_minus2'],
    right_on = ['station', 'date', 'hour'],
    how      = 'left'
).drop(columns=['station', 'hour'])

print(f"  Match rate: {bts_merged['dest_temp_f'].notna().mean():.1%}")
print()

STEP 8: MERGING DESTINATION WEATHER
  Match rate: 96.6%



In [17]:
# ============================================================
# STEP 9: Validate and save
# ============================================================
print("=" * 60)
print("STEP 9: VALIDATION & OUTPUT")
print("=" * 60)

print(f"\n  📐 Final shape: {bts_merged.shape}")

print(f"\n  🌤️  Origin match rate:  "
      f"{bts_merged['origin_temp_f'].notna().mean():.1%}")
print(f"  🌤️  Dest match rate:    "
      f"{bts_merged['dest_temp_f'].notna().mean():.1%}")

weather_cols = [c for c in bts_merged.columns
                if c.startswith('origin_') or c.startswith('dest_')]
print(f"\n  🆕 Weather columns added: {len(weather_cols)}")

if 'WEATHER_DELAY' in bts_merged.columns:
    wx_delay = bts_merged[bts_merged['WEATHER_DELAY'] > 0]
    if len(wx_delay) > 0:
        severe_delayed = wx_delay['origin_severe_weather'].mean()
        severe_overall = bts_merged['origin_severe_weather'].mean()
        print(f"\n  🔍 Sanity check:")
        print(f"      Severe weather rate (delayed flights): {severe_delayed:.2f}")
        print(f"      Severe weather rate (all flights):     {severe_overall:.2f}")
        if severe_delayed > severe_overall:
            print("      ✅ Looks correct!")

bts_merged.to_parquet(OUTPUT_FILE, index=False)

print(f"\n  💾 Saved: {OUTPUT_FILE}")
print(f"     Size: {os.path.getsize(OUTPUT_FILE) / (1024**2):.1f} MB")

print("\n" + "=" * 60)
print("✅ PIPELINE COMPLETE")
print("=" * 60)

STEP 9: VALIDATION & OUTPUT

  📐 Final shape: (45968068, 52)

  🌤️  Origin match rate:  96.5%
  🌤️  Dest match rate:    96.6%

  🆕 Weather columns added: 30

  💾 Saved: bts_with_weather.parquet
     Size: 1497.3 MB

✅ PIPELINE COMPLETE


## Section 3: Holiday & Weekend Flags

Adds `is_weekend` and `is_holiday` (US federal holidays ±1 day) flags.

In [18]:
df = pd.read_parquet(os.path.join(DRIVE_PATH, 'bts_with_weather.parquet'))
print(f'Loaded: {df.shape}')

FileNotFoundError: [Errno 2] No such file or directory: '/Users/sripriya/ucsd/MDS/CapstoneProject/AirlineArrivalDelay/data/bts_with_weather.parquet'

In [ ]:
!pip install holidays --quiet

In [ ]:
# is_weekend: DayOfWeek 6=Saturday, 7=Sunday (DOT convention)
df['is_weekend'] = df['DayOfWeek'].isin([6, 7]).astype(int)
# is_holiday: US federal holidays ±1 day window
us_holidays = holidays.US(years=range(2018, 2025))
holiday_dates = set(us_holidays.keys())
holiday_window = (
    holiday_dates
    | {d + timedelta(days=1) for d in holiday_dates}
    | {d - timedelta(days=1) for d in holiday_dates}
)
df['is_holiday'] = df['FlightDate'].dt.date.isin(holiday_window).astype(int)
print("is_weekend value counts:")
print(df['is_weekend'].value_counts())
print("\nis_holiday value counts:")
print(df['is_holiday'].value_counts())
df[['FlightDate', 'DayOfWeek', 'is_weekend', 'is_holiday']].head(10)

In [ ]:
# Save final output
final_path = os.path.join(DRIVE_PATH, 'bts_with_weather_holiday.parquet')
df.to_parquet(final_path, index=False)
print(f'Saved: {final_path}  Shape: {df.shape}')
# Delete intermediate files
print('\nCleaning up intermediate files...')
cache_dir = os.path.join(DRIVE_PATH, 'weather_cache')
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print('  Deleted weather_cache/')
for fname in ['bts_with_weather.parquet', 'flights_2018_2024_v2.parquet', 'weather_hourly_all_airports.parquet']:
    fpath = os.path.join(DRIVE_PATH, fname)
    if os.path.exists(fpath):
        os.remove(fpath)
        print(f'  Deleted {fname}')
print('\nDone. data/ now contains only: bts_with_weather_holiday.parquet')
